# Ground state search with a 2D autoregressive RWKV network

This notebook shows how to use `jVMC.nets.LogRWKV2DAutoregressiveJVMC` (alias `RWKV2D`), the two-dimensional version of the autoregressive RWKV network of the 1D RWKV example, on the 2D transverse-field Ising model.

In [1]:
import numpy as np
import jax.numpy as jnp

import jVMC
import jVMC.nets as nets
import jVMC.operator.discrete as op
import jVMC.sampler as sampler
from jVMC.vqs import NQS

Running in single-node mode (JVMC_USE_DISTRIBUTED not set)


/Users/som30090/code/Custom_jVMC/jVMC/solver/util.py:10: JaxMgWarning: Unsupported platform darwin, only Linux is supported. Non-Linux only works for docs.
  import jaxmg
/Users/som30090/code/Custom_jVMC/jVMC/solver/util.py:10: JaxMgWarning: No GPUs found, only use this mode for testing or generating documentation.
  import jaxmg


## Hamiltonian

$$
H = -\sum_{\langle i,j\rangle} \sigma^z_i \sigma^z_j - g \sum_i \sigma^x_i ,
$$

on an $L_x \times L_y$ lattice with periodic boundary conditions and $g = 1$. A configuration has shape `(Ly, Lx)`, and the operators act on the sites of the flattened configuration, so the site $(x, y)$ has index `y * Lx + x`.

The bonds are collected in a set, so each pair of neighbours is coupled once. On a $2 \times 2$ lattice the bond across the boundary connects the same two sites as the direct one, so there are only 4 distinct bonds. Increase `Lx` and `Ly` for a genuinely two-dimensional lattice; the exact diagonalisation at the end is only practical for a few more sites.

In [2]:
Lx, Ly = 2, 2
L = Lx * Ly
g = 1.0

def site(x, y):
    return y * Lx + x

bonds = set()
for y in range(Ly):
    for x in range(Lx):
        i = site(x, y)
        bonds.add(tuple(sorted((i, site((x + 1) % Lx, y)))))
        bonds.add(tuple(sorted((i, site(x, (y + 1) % Ly)))))

hamiltonian = 0
for i, j in sorted(bonds):
    hamiltonian += -op.SigmaZ(i) * op.SigmaZ(j)
for i in range(L):
    hamiltonian += -g * op.SigmaX(i)

## 2D autoregressive RWKV network

As in 1D, the wave function is a product of normalised conditional probabilities, $\psi(s) = \prod_i \sqrt{p(s_i \mid s_{<i})}\, e^{i\phi(s)}$, so it can be sampled exactly. The sites are visited row by row, and the hidden state of each site is computed from its own input and the hidden states of its left and upper neighbours, which keeps the two-dimensional structure of the lattice while remaining causal.

The main arguments are:

- `L`, `l_hil_dim`: number of sites and local Hilbert space dimension,
- `patch_size`: number of sites per token; `grid_Lx * grid_Ly` must equal the number of tokens, `L // patch_size`,
- `num_layers`, `num_heads`, `embedding_size`, `hidden_size`: size of the network,
- `flag_phase`: whether the network also learns a phase $\phi(s)$; without it, the wave function is positive,
- `param_dtype`, `compute_dtype`: data types of the parameters and of the computation.

With $2^4 = 16$ basis states we use the `ExactSampler`. For larger systems, use `jVMC.sampler.MCSampler`, which draws independent samples directly from the network (see the 1D RWKV example).

In [3]:
net = nets.LogRWKV2DAutoregressiveJVMC(
    L=L,
    l_hil_dim=2,
    patch_size=1,
    grid_Lx=Lx,
    grid_Ly=Ly,
    hidden_size=8,
    num_heads=1,
    num_layers=1,
    embedding_size=8,
    flag_phase=True,
    param_dtype=jnp.float32,
    compute_dtype=jnp.float32,
)
psi = NQS(net, (Ly, Lx), batch_size=2**L, seed=1234)
exact_sampler = sampler.ExactSampler(psi)
print("Number of parameters:", psi.num_parameters)

Number of parameters: 1744


## Optimisation with MinSR

As in the ViT and 1D RWKV examples, we use `MinSR`, which solves a linear system whose size is the number of samples (16) instead of the number of parameters (1744).

In [4]:
loss_function = jVMC.objective_function.Observable(hamiltonian)
stepper = jVMC.stepper.Euler(5e-2)
solver = jVMC.solver.Pinv(pinv_cutoff=1e-8)
opt = jVMC.optimizer.MinSR(exact_sampler, psi, solver=solver, diagonal_shift=1e-3)

out = opt.ground_state_search(500, loss_function, stepper)

100%|██████████| 500/500 [00:06<00:00, 81.85it/s, E=-5.2263e+00+3.3881e-21j ± 2.1739e-05 (Var = 2.0516e-09)] 

Recorded timings:
    - sampling: 0.000188s
    - compute objective function and gradient: 0.000846s
    - solve: 0.000671s


## Comparison with exact diagonalisation

The `ExactSampler` stores the wave function of the last `sample()` call and does not update it when the parameters change. The last optimisation step updates the parameters after the measurement, so we call `sample()` again before measuring the final energy. The exact ground state energy is obtained by diagonalising the $16 \times 16$ Hamiltonian matrix.

In [5]:
exact_sampler.sample()
energy = exact_sampler(hamiltonian).mean.real.item()
H_dense = hamiltonian.to_dense(exact_sampler.basis)
exact_energy = np.linalg.eigvalsh(H_dense)[0]

print(f"VMC energy:     {energy:.8f}")
print(f"Exact energy:   {exact_energy:.8f}")
print(f"Relative error: {abs((energy - exact_energy) / exact_energy):.1e}")

VMC energy:     -5.22625185
Exact energy:   -5.22625186
Relative error: 9.4e-10
